# Feature Engineering
Create a Dataframe for model input:
- Feature creation
- Daily aggregation of features
- Vizualization

# Import Libraries and Data

In [1]:
# --- Standard Libraries ---
import os
import re
from collections import Counter

# --- Data Science ---
import pandas as pd
import numpy as np

# --- NLP ---
import nltk

# --- Transformers ---
import torch
from torch.nn.functional import sigmoid
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertTokenizer,
    BertForSequenceClassification
)
from scipy.special import softmax

# --- Utils ---
from tqdm.auto import tqdm
from IPython.display import display

# --- Setup ---
tqdm.pandas()
nltk.download("punkt")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Kontrollvariablen

In [2]:
# Uses shorter time periods for testing purposes
TEST = False # default = False

# Wenn nur einzelne Spalten/Features hinzugefügt werden sollen, bitte unten den Block 'neue Features zur CSV hinzufügen' entsprechend anpassen. Die bestehende final_daily_df csv wird dann in ein df geladen und die neuen Spalten werden dazugemerged und die csv wieder abgespeichert.
einzelne_features_zur_bestehenden_CSV_hinzufügen = False # default = False

# wenn True, wird die final_daily_df CSV ganz neu zusammengestellt.
vollstaendige_neuerstellung_der_csv = True  # default = False

In [3]:
musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["isRetweet"] = df["isRetweet"].astype(str).str.lower()
    df["possiblySensitive"] = df["possiblySensitive"].astype(str).str.lower()
    df["fullText"] = df["fullText"].astype(str)

musk_twitter_data_nlp["text_raw"] = musk_twitter_data_nlp["text_raw"].astype(str)
musk_twitter_data_nlp["text_lemmatized"] = musk_twitter_data_nlp["text_lemmatized"].astype(str)
musk_twitter_data_nlp["quote_and_original_lemmatized"] = musk_twitter_data_nlp["quote_and_original_lemmatized"].astype(str)

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["date"] = df["createdAt"].dt.date

if TEST:
    start_date = pd.to_datetime("2025-04-01").date()
else:
    start_date = pd.to_datetime("2015-01-01").date()

end_date = musk_twitter_data_all["date"].max()

mask_all = (musk_twitter_data_all["date"] >= start_date) & (musk_twitter_data_all["date"] <= end_date)
musk_twitter_data_all = musk_twitter_data_all.loc[mask_all].reset_index(drop=True)

mask_nlp = (musk_twitter_data_nlp["date"] >= start_date) & (musk_twitter_data_nlp["date"] <= end_date)
musk_twitter_data_nlp = musk_twitter_data_nlp.loc[mask_nlp].reset_index(drop=True)

final_daily_df_base = pd.DataFrame({
    'date': pd.date_range(start=start_date, end=end_date)
})
final_daily_df_base["date"] = final_daily_df_base["date"].dt.date 

print("NLPTweets:", musk_twitter_data_nlp.shape, "AllTweets:", musk_twitter_data_all.shape)
musk_twitter_data_nlp.info()
musk_twitter_data_all.info()

C:\Users\malte\AppData\Local\Temp\ipykernel_4896\3683667789.py:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
C:\Users\malte\AppData\Local\Temp\ipykernel_4896\3683667789.py:2: DtypeWarning: Columns (11,16,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])


NLPTweets: (45150, 35) AllTweets: (54023, 30)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45150 entries, 0 to 45149
Data columns (total 35 columns):
 #   Column                         Non-Null Count  Dtype              
---  ------                         --------------  -----              
 0   id                             45150 non-null  int64              
 1   url                            45150 non-null  object             
 2   twitterUrl                     45150 non-null  object             
 3   fullText                       45150 non-null  object             
 4   retweetCount                   45089 non-null  float64            
 5   replyCount                     44543 non-null  float64            
 6   likeCount                      45089 non-null  float64            
 7   quoteCount                     44529 non-null  float64            
 8   viewCount                      28556 non-null  float64            
 9   createdAt                      45150 non-null  d

# Tweet activity
New features:
- Number of tweets per day

In [4]:
tweet_counts_daily = (
    musk_twitter_data_all
    .groupby("date")
    .size()
    .reset_index(name="tweet_count")
)

# Engagement metrics
- like_count
- quoted_count
- retweet_count
- view_count

In [5]:
# Engagement metrics calculation: Like, Quote, Retweet, Reply counts per day
engagement_metrics = (
    musk_twitter_data_all
    .groupby('date')[['likeCount', 'quoteCount', 'retweetCount', 'replyCount']]
    .sum()
    .astype(int)
    .reset_index()
)


# Sentiment Analysis

New Features:
- Poitve, Neutral ans Negative percentage of posts
- Polarization: Tweets with pos/neg > 0,6

In [ ]:
# Tweet sentiment analysis
# Model: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def preprocess(text):
    return text.replace("\n", " ").strip()

def get_sentiment_probs(text):
    text = preprocess(text)
    tokens = tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.cpu().numpy()[0])
    return {
        "sentiment": ['negative', 'neutral', 'positive'][probs.argmax()],
        "neg": probs[0],
        "neu": probs[1],
        "pos": probs[2],
    }

def polarized_label(row):
    return "polarized" if max(row["pos"], row["neg"]) > 0.6 else "not_polarized"

results = musk_twitter_data_nlp["text_raw"].progress_apply(get_sentiment_probs).apply(pd.Series)
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp, results], axis=1)
max_sent = musk_twitter_data_nlp[["pos", "neg"]].max(axis=1)
musk_twitter_data_nlp["sentiment_polarity"] = np.where(max_sent > 0.6, "polarized", "not_polarized")

# 1) Unweighted daily aggregation (mean) with only the “polarized” share
sentiment_avg = (musk_twitter_data_nlp.groupby("date")[["neg", "neu", "pos"]].mean().reset_index())
nlp_counts = (musk_twitter_data_nlp.groupby("date").size().reset_index(name="nlp_tweet_count"))

polar_mean = (musk_twitter_data_nlp.groupby("date")["sentiment_polarity"].apply(lambda s: (s == "polarized").mean()).reset_index(name="polarized"))

sentiment_daily = (
    sentiment_avg
    .merge(nlp_counts, on="date", how="left")
    .merge(polar_mean,  on="date", how="left")
)


# 2) Weighted daily aggregation
weighted_sums = (
    musk_twitter_data_nlp
    .assign(
        neg_w = lambda df: df["neg"] * df["engagement_index"],
        neu_w = lambda df: df["neu"] * df["engagement_index"],
        pos_w = lambda df: df["pos"] * df["engagement_index"],
    )
    .groupby("date")
    .agg(
        neg_w_sum        = ("neg_w", "sum"),
        neu_w_sum        = ("neu_w", "sum"),
        pos_w_sum        = ("pos_w", "sum"),
        total_engagement = ("engagement_index", "sum"),
    )
    .reset_index()
    .assign(
        neg = lambda df: df["neg_w_sum"] / df["total_engagement"],
        neu = lambda df: df["neu_w_sum"] / df["total_engagement"],
        pos = lambda df: df["pos_w_sum"] / df["total_engagement"],
    )
    .drop(columns=["neg_w_sum", "neu_w_sum", "pos_w_sum"])
)

# 2b) Weighted polarization (only “polarized”)
polar_weighted = (
    musk_twitter_data_nlp
    .groupby(["date", "sentiment_polarity"])["engagement_index"]
    .sum()
    .reset_index(name="eng_w_sum")
    .pivot(index="date", columns="sentiment_polarity", values="eng_w_sum")
    .fillna(0)
    .reset_index()
    .merge(weighted_sums[["date", "total_engagement"]], on="date", how="left")
    .assign(polarized=lambda df: df["polarized"] / df["total_engagement"])
    [["date", "polarized"]]
)

# 2c) Final weighted daily DataFrame
sentiment_daily_weighted = (
    weighted_sums[["date", "neg", "neu", "pos"]]
    .merge(polar_weighted, on="date", how="left")
    .merge(nlp_counts,        on="date", how="left")
)


  0%|          | 0/45150 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


# Emotions & Personality

New Features:
- Ekman Emotions: anger, disgust, fear, joy, neutral, sadness, surprise
- Big 5 personality traits: Extroversion, Neuroticism, Agreeableness, Conscientiousness, Openness

In [7]:
# Ekman Emotionen
# Model: https://huggingface.co/j-hartmann/emotion-english-distilroberta-base
model_name = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# Helper that returns a dict of probabilities
def get_emotions(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        logits = model(**tokens).logits
    probs = softmax(logits.numpy()[0])
    return dict(zip(emotion_labels, probs))

# Apply to every tweet
print("Calculating emotion probabilities...")
emotion_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_emotions).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), emotion_probs],axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily emotions...")
emotion_daily = (musk_twitter_data_nlp.groupby('date')[emotion_labels].mean().reset_index())

# Weighted daily aggregation
print("Aggregating daily emotions (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{emo}_w": lambda df, emo=emo: df[emo] * df["engagement_index"]
        for emo in emotion_labels
    })
    .groupby("date")
    .agg(
        **{f"{emo}_w_sum": (f"{emo}_w", "sum") for emo in emotion_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

emotion_daily_weighted = (
    weighted_sums
    .assign(**{
        emo: lambda df, emo=emo: df[f"{emo}_w_sum"] / df["total_engagement"]
        for emo in emotion_labels
    })
    [["date", *emotion_labels]]
)
print("Done!")

Calculating emotion probabilities...


  0%|          | 0/45115 [00:00<?, ?it/s]

Aggregating daily emotions...
Aggregating daily emotions (weighted)...
Done!


In [8]:
# Big Five Personality Traits
# Model: https://huggingface.co/Minej/bert-base-personality
tokenizer = BertTokenizer.from_pretrained("Minej/bert-base-personality")
model = BertForSequenceClassification.from_pretrained("Minej/bert-base-personality")

personality_labels = ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']

def get_personality(text):
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    probs = sigmoid(outputs.logits).squeeze().numpy()
    return dict(zip(personality_labels, probs))

# Apply to every tweet
print("Calculating personality traits...")
personality_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_personality).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), personality_probs], axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily personality...")
personality_daily = (musk_twitter_data_nlp.groupby('date')[personality_labels].mean().reset_index())

# Weighted daily aggregation
print("Aggregating daily personality (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{pers}_w": lambda df, pers=pers: df[pers] * df["engagement_index"]
        for pers in personality_labels
    })
    .groupby("date")
    .agg(
        **{f"{pers}_w_sum": (f"{pers}_w", "sum") for pers in personality_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

personality_daily_weighted = (
    weighted_sums
    .assign(**{
        pers: lambda df, pers=pers: df[f"{pers}_w_sum"] / df["total_engagement"]
        for pers in personality_labels
    })
    [["date", *personality_labels]]
)
print("Done!")

Calculating personality traits...


  0%|          | 0/45115 [00:00<?, ?it/s]

Aggregating daily personality...
Aggregating daily personality (weighted)...
Done!


# Topic and word counts

New Features: 
- Daily Word counts
    - Rationale of Definition of words:
        - Company/ticker terms (e.g. tesla, tsla, spacex) capture direct references to publicly traded entities.
        - Product names (e.g. model, cybertruck, starship) often precede news that can move stock prices.
        - Crypto tokens (e.g. bitcoin, dogecoin, ethereum, crypto) map to Musk-driven volatility in the digital-asset markets
        - Financial keywords (e.g. stock, market, price, profit, loss, revenue) directly signal earnings or valuation discussions.
        - Macro terms (e.g. inflation, interest) reflect broader economic commentary that can sway sentiment.
        - Action verbs (buy, sell) often presage trading intent or recommendations.
- Topics of posts

In [9]:
# Words
musk_twitter_data_nlp['text'] = np.where(
    musk_twitter_data_nlp['isQuote'],
    musk_twitter_data_nlp['quote_and_original_lemmatized'],
    musk_twitter_data_nlp['text_lemmatized']
)
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|@\S+|[^a-z\s]", "", text)
    return text.split()

all_tokens = musk_twitter_data_nlp['text'].dropna().apply(tokenize)
flat_tokens = [token for sublist in all_tokens for token in sublist]
word_counts = Counter(flat_tokens)
word_counts = (
    pd.DataFrame(word_counts.items(), columns=["word", "count"])
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

top20 = [
    'tesla', 'stock', 'market', 'price', 'profit', 'loss', 'revenue',
    'inflation', 'interest', 'bitcoin', 'dogecoin', 'crypto', 'ethereum',
    'spacex', 'model', 'cybertruck', 'starship', 'buy', 'sell'
]

top_word_df = musk_twitter_data_nlp.dropna(subset=['text']).copy()
top_word_df['tokens'] = top_word_df['text'].apply(tokenize)
top_word_df = top_word_df.explode('tokens')
top_word_df['tokens'] = top_word_df['tokens'].replace({'tsla': 'tesla'})

top_word_df = top_word_df[top_word_df['tokens'].isin(top20)].copy()

daily_word_counts = (
    top_word_df
    .groupby(['date','tokens'])
    .size()
    .unstack(fill_value=0)
)

daily_word_counts = daily_word_counts.reindex(
    columns=top20,
    fill_value=0
).sort_index()

In [10]:
# Topic Analysis
# Model: https://huggingface.co/cardiffnlp/tweet-topic-21-multi
import torch
import numpy as np
from scipy.special import softmax
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import logging

# Set up logging to catch warnings
logging.basicConfig(level=logging.WARNING)

model_name = "cardiffnlp/tweet-topic-21-multi"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

topic_labels = [
    "arts_culture", "business_entrepreneurs", "celebrity_pop_culture",
    "diaries_daily_life", "family", "fashion_style", "film_tv_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_educational",
    "music", "news_social_concern", "other_hobbies", "relationships",
    "science_technology", "sports", "travel_adventure", "youth_student_life"
]

def get_topics(text):
    """
    Get topic probabilities for a given text.
    Fixed to handle tensor size mismatches and edge cases.
    """
    try:
        # Handle empty or None text
        if not text or pd.isna(text):
            return {label: 0.0 for label in topic_labels}
        
        # Convert to string and clean
        text = str(text).strip()
        if not text:
            return {label: 0.0 for label in topic_labels}
        
        # Tokenize with proper parameters
        tokens = tokenizer(
            text, 
            truncation=True, 
            padding=True, 
            max_length=512,  # Explicit max length
            return_tensors="pt"
        )
        
        # Run inference
        with torch.no_grad():
            output = model(**tokens)
        
        # Get probabilities
        logits = output.logits.numpy()[0]
        probs = softmax(logits)
        
        return dict(zip(topic_labels, probs))
    
    except Exception as e:
        print(f"Error processing text: {str(e)[:100]}...")
        # Return uniform distribution as fallback
        uniform_prob = 1.0 / len(topic_labels)
        return {label: uniform_prob for label in topic_labels}

# Apply to every tweet with better error handling
print("Calculating topic probabilities...")
try:
    # Use tqdm if available, otherwise regular apply
    if hasattr(musk_twitter_data_nlp['text'], 'progress_apply'):
        topic_scores = musk_twitter_data_nlp['text'].progress_apply(get_topics).apply(pd.Series)
    else:
        topic_scores = musk_twitter_data_nlp['text'].apply(get_topics).apply(pd.Series)
    
    # Ensure all topic labels are present as columns
    for label in topic_labels:
        if label not in topic_scores.columns:
            topic_scores[label] = 0.0
    
    # Reorder columns to match topic_labels
    topic_scores = topic_scores[topic_labels]
    
except Exception as e:
    print(f"Error during topic analysis: {e}")
    # Create fallback dataframe with uniform probabilities
    uniform_prob = 1.0 / len(topic_labels)
    topic_scores = pd.DataFrame(
        {label: [uniform_prob] * len(musk_twitter_data_nlp) for label in topic_labels}
    )

# Append those new columns back onto your original DF
print("Merging topic scores with original data...")
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), topic_scores], axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily topics...")
topics_daily = (
    musk_twitter_data_nlp.groupby('date')[topic_labels]
    .mean()
    .reset_index()
)

# Weighted daily aggregation
print("Aggregating daily topics (weighted)...")
try:
    # Create weighted columns more safely
    weighted_data = musk_twitter_data_nlp.copy()
    
    # Add weighted columns
    for top in topic_labels:
        weighted_data[f"{top}_w"] = weighted_data[top] * weighted_data["engagement_index"]
    
    # Aggregate
    weighted_sums = (
        weighted_data
        .groupby("date")
        .agg(
            **{f"{top}_w_sum": (f"{top}_w", "sum") for top in topic_labels},
            total_engagement=("engagement_index", "sum"),
        )
        .reset_index()
    )
    
    # Calculate weighted averages
    topics_daily_weighted = weighted_sums[["date"]].copy()
    for top in topic_labels:
        # Avoid division by zero
        topics_daily_weighted[top] = np.where(
            weighted_sums["total_engagement"] > 0,
            weighted_sums[f"{top}_w_sum"] / weighted_sums["total_engagement"],
            0.0
        )
    
except Exception as e:
    print(f"Error during weighted aggregation: {e}")
    print("Using unweighted aggregation as fallback...")
    topics_daily_weighted = topics_daily.copy()

print("Done!")

# Optional: Display some statistics
print(f"\nTopic analysis completed for {len(musk_twitter_data_nlp)} tweets")
print(f"Date range: {topics_daily['date'].min()} to {topics_daily['date'].max()}")
print(f"Topic columns: {topic_labels}")

Calculating topic probabilities...


  0%|          | 0/45115 [00:00<?, ?it/s]

Merging topic scores with original data...
Aggregating daily topics...
Aggregating daily topics (weighted)...
Done!

Topic analysis completed for 45115 tweets
Date range: 2015-01-05 to 2025-04-13
Topic columns: ['arts_culture', 'business_entrepreneurs', 'celebrity_pop_culture', 'diaries_daily_life', 'family', 'fashion_style', 'film_tv_video', 'fitness_&_health', 'food_&_dining', 'gaming', 'learning_educational', 'music', 'news_social_concern', 'other_hobbies', 'relationships', 'science_technology', 'sports', 'travel_adventure', 'youth_student_life']


In [11]:
""" # Topic Analysis
# Model: https://huggingface.co/cardiffnlp/tweet-topic-21-multi
model_name = "cardiffnlp/tweet-topic-21-multi"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

topic_labels = [
    "arts_culture", "business_entrepreneurs", "celebrity_pop_culture",
    "diaries_daily_life", "family", "fashion_style", "film_tv_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_educational",
    "music", "news_social_concern", "other_hobbies", "relationships",
    "science_technology", "sports", "travel_adventure", "youth_student_life"
]

def get_topics(text):
    tokens = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.numpy()[0])
    return dict(zip(topic_labels, probs))

# Apply to every tweet
print("Calculating topic probabilities...")
topic_scores = musk_twitter_data_nlp['text'].progress_apply(get_topics).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), topic_scores], axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily topics...")
topics_daily = (musk_twitter_data_nlp.groupby('date')[topic_labels].mean().reset_index())

# Weighted daily aggregation
print("Aggregating daily topics (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{top}_w": lambda df, top=top: df[top] * df["engagement_index"]
        for top in topic_labels
    })
    .groupby("date")
    .agg(
        **{f"{top}_w_sum": (f"{top}_w", "sum") for top in topic_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

# 2b) Traits wieder auf die Originalnamen zurückskalieren
topics_daily_weighted = (
    weighted_sums
    .assign(**{
        top: lambda df, top=top: df[f"{top}_w_sum"] / df["total_engagement"]
        for top in topic_labels
    })
    [["date", *topic_labels]]
)
print("Done!") """

' # Topic Analysis\n# Model: https://huggingface.co/cardiffnlp/tweet-topic-21-multi\nmodel_name = "cardiffnlp/tweet-topic-21-multi"\ntokenizer = AutoTokenizer.from_pretrained(model_name)\nmodel = AutoModelForSequenceClassification.from_pretrained(model_name)\n\ntopic_labels = [\n    "arts_culture", "business_entrepreneurs", "celebrity_pop_culture",\n    "diaries_daily_life", "family", "fashion_style", "film_tv_video",\n    "fitness_&_health", "food_&_dining", "gaming", "learning_educational",\n    "music", "news_social_concern", "other_hobbies", "relationships",\n    "science_technology", "sports", "travel_adventure", "youth_student_life"\n]\n\ndef get_topics(text):\n    tokens = tokenizer(text, truncation=True, padding=True, return_tensors="pt")\n    with torch.no_grad():\n        output = model(**tokens)\n    probs = softmax(output.logits.numpy()[0])\n    return dict(zip(topic_labels, probs))\n\n# Apply to every tweet\nprint("Calculating topic probabilities...")\ntopic_scores = musk_

In [12]:
topics_daily_weighted.head()
topics_daily.head()

,date,arts_culture,business_entrepreneurs,celebrity_pop_culture,diaries_daily_life,family,fashion_style,film_tv_video,fitness_&_health,food_&_dining,gaming,learning_educational,music,news_social_concern,other_hobbies,relationships,science_technology,sports,travel_adventure,youth_student_life
0,2015-01-05,0.001533,0.005354,0.004738,0.041399,0.000540,0.000404,0.009824,0.000476,0.000696,0.001542,0.003858,0.001551,0.796910,0.011698,0.001117,0.104953,0.002432,0.009344,0.001631
1,2015-01-06,0.002646,0.004239,0.005932,0.042633,0.000920,0.000318,0.006809,0.001057,0.000683,0.001021,0.004372,0.001815,0.624436,0.008709,0.001803,0.281972,0.005848,0.003172,0.001614
2,2015-01-10,0.008797,0.038474,0.006918,0.189333,0.001543,0.002122,0.072397,0.001818,0.002508,0.009192,0.011615,0.022416,0.218480,0.045593,0.002827,0.310021,0.011981,0.041257,0.002710
3,2015-01-11,0.000345,0.021637,0.000819,0.000858,0.000303,0.000242,0.001516,0.001210,0.000663,0.000621,0.005107,0.000580,0.067496,0.001125,0.000365,0.893970,0.000827,0.000938,0.001376
4,2015-01-12,0.014893,0.006263,0.010204,0.018899,0.002197,0.001489,0.012717,0.002197,0.001750,0.004554,0.029326,0.012859,0.104657,0.016331,0.002606,0.713392,0.003677,0.035945,0.006044


## Additional Features to consider/ ToDos

- ToDo Tweet-Typ (z. B. Meme, Information, Ankündigung, Meinung, Engagement)
    - Studien zeigen, dass z. B. Meme-Posts und ironische Tweets besonders starke Kursreaktionen auslösen 
    - Bei Musk besonders relevant, da sein Kommunikationsstil sich im Zeitverlauf stark verändert hat 


Aus Termin mit Peter:
- Einflussreiche weitere Personen: Kann man ggf auch aus quotes nehmen, ist mir nicht mehr ganz klar was er wollte.
- Quotes mit einbeziehen, Quote dataset enthält die texte der Quotes -> Einbeziehen, höhere Genauigkeit bei eg toics

# Create final Daily DF
One can just add Features to the existing Dataframe or create a new Final Daily Df
### Add new Features to existing dataframe

In [13]:
if einzelne_features_zur_bestehenden_CSV_hinzufügen:
    # 1. Final-Dataset laden mit geparster Datumsspalte
    final_daily_df = pd.read_csv("Data/twitter_data/processed/final_daily_df.csv", parse_dates=["date"])

    # 2. Platzhaltervariable für zusätzliche Feature-DataFrames
    # Beispiel: zusatz_feature_dfs = [df_neues_feature_1, df_neues_feature_2, ...]
    zusatz_feature_dfs = [
        
        # HIER DIE OBEN ERSTELLTEN NEUEN SPALTEN (inkl. 'date' spalte) AUFLISTEN
    ]

    # 3. Iterativ mergen
    for feature_df in zusatz_feature_dfs:
        
        feature_df["date"] = pd.to_datetime(feature_df["date"])
        
        # Prüfen auf doppelte Spalten (außer 'date')
        doppelte = [col for col in feature_df.columns if col != "date" and col in final_daily_df.columns]
        if doppelte:
            raise ValueError(f"Die folgenden Spalten sind bereits in final_daily_df vorhanden und sollten evtl. nicht erneut gemerged werden: {doppelte}")

        # Merge auf 'date'
        final_daily_df = pd.merge(final_daily_df, feature_df, on="date", how="left")

    # 4. Ergebnis zurückschreiben
    final_daily_df.to_csv("Data/twitter_data/processed/final_daily_df.csv", index=False)


### Merge and Create final df
Merge the daily dfs in one new dataframe and create csv (creates a weighted and an unweighted version)

In [14]:
# Merge with complete date, fill missing days with zero
if vollstaendige_neuerstellung_der_csv:
    # Unweighted final daily DataFrame
    final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    final_daily_df = final_daily_df.merge(engagement_metrics, on="date", how="left")
    final_daily_df["tweet_count"] = final_daily_df["tweet_count"].astype(int)
    final_daily_df = final_daily_df.merge(sentiment_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(emotion_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(personality_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(daily_word_counts, on="date", how="left")
    final_daily_df = final_daily_df.merge(topics_daily, on="date", how="left")
    final_daily_df["no_tweets"] = (final_daily_df["tweet_count"] == 0).astype(int)
    display(final_daily_df.info())
    display(final_daily_df.head())
    # Weighted final daily DataFrame
    weighted_final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    weighted_final_daily_df = weighted_final_daily_df.merge(engagement_metrics, on="date", how="left")
    weighted_final_daily_df["tweet_count"] = weighted_final_daily_df["tweet_count"].astype(int)
    weighted_final_daily_df = weighted_final_daily_df.merge(sentiment_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(emotion_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(personality_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(daily_word_counts, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(topics_daily_weighted, on="date", how="left")
    weighted_final_daily_df["no_tweets"] = (weighted_final_daily_df["tweet_count"] == 0).astype(int)

    display(weighted_final_daily_df.info())
    display(weighted_final_daily_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3756 entries, 0 to 3755
Data columns (total 62 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    3756 non-null   object 
 1   tweet_count             3756 non-null   int64  
 2   likeCount               3056 non-null   float64
 3   quoteCount              3056 non-null   float64
 4   retweetCount            3056 non-null   float64
 5   replyCount              3056 non-null   float64
 6   neg                     3010 non-null   float32
 7   neu                     3010 non-null   float32
 8   pos                     3010 non-null   float32
 9   nlp_tweet_count         3010 non-null   float64
 10  polarized               3010 non-null   float64
 11  anger                   3010 non-null   float32
 12  disgust                 3010 non-null   float32
 13  fear                    3010 non-null   float32
 14  joy                     3010 non-null   

None

,date,tweet_count,likeCount,quoteCount,retweetCount,replyCount,neg,neu,pos,nlp_tweet_count,...,learning_educational,music,news_social_concern,other_hobbies,relationships,science_technology,sports,travel_adventure,youth_student_life,no_tweets
0,2015-01-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2015-01-02,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2015-01-03,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2015-01-04,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2015-01-05,2,3575.0,3.0,3625.0,400.0,0.020146,0.926847,0.053006,2.0,...,0.003858,0.001551,0.79691,0.011698,0.001117,0.104953,0.002432,0.009344,0.001631,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3756 entries, 0 to 3755
Data columns (total 62 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    3756 non-null   object 
 1   tweet_count             3756 non-null   int64  
 2   likeCount               3056 non-null   float64
 3   quoteCount              3056 non-null   float64
 4   retweetCount            3056 non-null   float64
 5   replyCount              3056 non-null   float64
 6   neg                     3010 non-null   float64
 7   neu                     3010 non-null   float64
 8   pos                     3010 non-null   float64
 9   polarized               3010 non-null   float64
 10  nlp_tweet_count         3010 non-null   float64
 11  anger                   3010 non-null   float64
 12  disgust                 3010 non-null   float64
 13  fear                    3010 non-null   float64
 14  joy                     3010 non-null   

None

,date,tweet_count,likeCount,quoteCount,retweetCount,replyCount,neg,neu,pos,polarized,...,learning_educational,music,news_social_concern,other_hobbies,relationships,science_technology,sports,travel_adventure,youth_student_life,no_tweets
0,2015-01-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2015-01-02,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2015-01-03,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2015-01-04,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2015-01-05,2,3575.0,3.0,3625.0,400.0,0.022617,0.932019,0.045364,0.0,...,0.001767,0.000717,0.909028,0.004948,0.000479,0.047566,0.001076,0.003952,0.000748,0


### Export

In [15]:
if vollstaendige_neuerstellung_der_csv:
    final_daily_df.to_csv(os.path.join('processed', 'final_daily_df.csv'), index=False)
    weighted_final_daily_df.to_csv(os.path.join('processed', 'weighted_final_daily_df.csv'), index=False)